<h1 style="text-align: center; font-size: bolder;"> PySpark </h1>

![Python+Spark=Pyspark](../images/pyspark.jpg)

**Authored by: Thandokuhle Brian Msane, Prof. Stephen Fashoto, and Bandile Malaza**

## Introduction

**Spark** is a unified analytical engine for large-scale data processing. It processes vast amounts of data in a distributed or parallel computing fashion. It uses a _cluster_ which is a group of computers (nodes or executors) that collaboratively processes large datasets using the Spark interface. In the cluster, we have a **master** and one or more **worker nodes**. The master manages the cluster, coordinates tasks, and schedule jobs. The worker nodes execute the tasks assigned to them by the master and they are actually responsible for computations and storing data in memory.

It is important to note that Spark is not the solution to every data problem. If you dealing with small datasets that can be processed by a single machine, it's not ideal to use Spark! Managing a distributed cluster comes with some overhead so if you're using a single node you are not basically getting any benefits but paying the price.


### Master and Slave Architecture

![Master&Slaves](../images/spark.jpg)

The master takes your instructions, translates them into Spark steps and processes them across the worker. One important characteristic of the master is the he is **lazy**. Every instruction you're providing to Spark can be classified into two categories which are **transformations** and **actions**. _Actions can be thought of as I/O operations like read and write_. Everything else except for actions are called transformations and these include adding a column to the DataFrame, aggregating data, and so on. So, with the lazy computation model we mean that Spark will avoid performing data work until an action triggers the compuation chain. Before that, the master will cache your instruction and this approach has many advantages when dealing with big datasets.

First, storing instructions **in-memory** takes less space than storing the intermediate data results. So, if you're performing many operation on a dataset and using data at each step you will quickly run out of storage yet you might not need the intermediate results.

Second, by having the list of tasks to be performed, the master can optimize the work between executors much more efficiently. The master uses information available at runtime like nodes with specific parts of the data. It can also _re-order and eliminate useless transformations_ if necessary.

Finally, during interactive development, you don't have to submit a huge block of commands and wait for compuation to happen. Instead you can iteratively build your chain of transformation, one at a time, and when you're ready to launch the computation, you can add an action and let Spark work its magic.

The summary of the benefits of PySpark are:

- `In-memory computation`: we can use all the memory that spark has by processing data directly from the system's RAM
- `Lazy evaluation`: transformations will not immediately evaluate/compute until an action triggers them
- `Fault Tolerant`: we have the ability to trace back all the transformations
- `Partition`: parititions and distributes the data on the cluster for faster processing. Each node get's its own portion of the data to work with.


### PySpark DataFrame

A DataFrame is a distributed dataset comprising data arranged in rows and columns with named attributes. It shares similarities with relational database tables or Python data frame but incorporates sophisticated organizations. "If you have a Python background, I would assume you already know what Pandas DataFrames are." A PySpark DataFrame is similar with Pandas DataFrame, with the exception that DataFrames are distributed in the cluster, as mentioned, and any operations in PySpark execute in parallel on all machines, whereas Pandas DataFrames store and operates in a single machine.

#### Is PySpark faster than Pandas?

PySpark is a distributed computing framework well-suited for processing large-scale datasets that exceed the memory capacity of a single machine. It can leverage paralled processing across a cluster of machines, enabling faster computations on massive datasets. On the other hand, Pandas, being a **single-machine** library, is optimized for smaller to medium-sized datasets taht can fit into memory. It typically performs well for data manipulation and analysis tasks on small to medium datasets.


## Starting a PySpark Session

- Common Keywords
  - `builder()` sets up the session
  - `appName()` is used to setup your application name. This helps us manage multiple PySpark applications
  - `getOrCreate()` retrieves a SparkSession object if it already exists, else creates a new one.
- Uncommon Keywords
  - `master()`. If you are running it on the cluster, you need to use your master name as an argument to the `master()` method. Usually, it would be either **YARN** (yet another resource negotiator) or **Mesos** depending on your cluster setup.
  - `local["*"]`. When operating in a standalone mode, specify `local[x]`, where `x` is an integer greater than 0, to determine the number of partitions for RDD. Ideally, set `x` to match the number of CPU cores available on your system for optimal performance.


Monitoring your PySpark jobs is an important part of developing a robust program. To help with this, PySpark provides some levels of logging. By default, **WARN** is used. The logging levels available are summarized into the following table.

<div align="center">

| Level | Description|
|-------|-------------|
| OFF | No logging at all|
|FATAL| Logs only fatal errors|
| ERROR| Shows fatal errors but also some recoverable ones|
|WARN | Adds warnings|
|INFO    | Gives runtime information
|DEBUG  | Provides debug information |
| TRACE | Will trace your jobs|
| ALL | Everything happening on your PySpark |
</div>

You can use the `setLogLevel` function in `spark.sparkContext` to set the logging level of your application if you want to override the default setting. This is shown below. Here, we are going to use the ERROR level to avoid getting even unnecessary information and warning but we want to pay attention to recoverable and fatal errors.


In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.ml.feature import Imputer
from pyspark.sql.functions import lit, col, when, avg, count, desc
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:

spark = SparkSession.builder\
    .appName("simpleapp")\
    .master("local[*]")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## Reading and Ingesting Data

For ingesting data, PySpark provides two main structures of data manipualtion which are RDD, _Resilient Distributed Dataset_, and DataFrame. The latter is a stricter version of the RDD and it processes data using rows while the former processes each rocord independently.

## Create a DataFrame

There are different methods for creating DataFrames in PySpark and these include:

- List of lists
- List of Tuples
- List of Dictionaries
- List of `pyspark.sql.Row()`
- Reading data from files usine `pyspark.sql.readwriter.DataFrameReader`

Use the `createDataFrame()` method from the SparkSession object to create the DataFrame object

### Example 1: List of Lists

In [3]:
data = [
    ["Metfula", 25, "Eswatini"],
    ["Elliot", 28, "Zimbabwe"],
    ["Aina", 30, "Namibia"]
]
df = spark.createDataFrame(data=data, schema=["Name", "Age", "Country"])

df.show() # display the content
df.printSchema()

+-------+---+--------+
|   Name|Age| Country|
+-------+---+--------+
|Metfula| 25|Eswatini|
| Elliot| 28|Zimbabwe|
|   Aina| 30| Namibia|
+-------+---+--------+

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Country: string (nullable = true)



For peeking into the data we can use the `show()` method. show() is an action since it performs the visible work of printing data on the screen.

It takes three optional arguments
- **n** can be set to any positive number which represents the number of rows to display. the default value is `20` for data frames with more rows otherwise, it shows the data frame with all rows.
- **truncate** is a boolean argument; when se to True, it will truance teh columns to displya only 20 characters otherwise it displays the whole length. It can also be set to a positive number which determines the specific number of characters to turncate to.
- **vertical** takes a Boolean value; if True, it displays each record as a small table

PySpark accepts the schema to be the a list containing the names of the columns and it automatically infers the data types like string, long, double, and so on.

Alhough Spark can automatically enfore schemas, sometimes it can misinterpret data types, particularly with complex data structures so, we have to manualy define the schema for accurate data handling. To do that, we make use of `StructType`, `StructField`, and different data types found in `pyspark.sql.types`.

#### List of Tuples

In [ ]:
schema = StructType([
    StructField('name', StringType(), True),
    StructField('club', StringType(), True),
    StructField('statement', StringType(), True),
])

data = [
    ("Fashoto", 'Man United', 'Eswatini is a very good country'),
    ("Malaza", 'Man City', 'PySpark is very nice'),
    ("Msane", 'Man United', 'Ubuntu is better than Windows, frankly')
]
df1 = spark.createDataFrame(data=data, schema=schema)

df1.show(n=3, truncate=True, vertical=True) # truncate strings more than 20 chars

df1.printSchema()

-RECORD 0-------------------------
 name      | Fashoto              
 club      | Man United           
 statement | Eswatini is a ver... 
-RECORD 1-------------------------
 name      | Malaza               
 club      | Man City             
 statement | PySpark is very nice 
-RECORD 2-------------------------
 name      | Msane                
 club      | Man United           
 statement | Ubuntu is better ... 

root
 |-- name: string (nullable = true)
 |-- club: string (nullable = true)
 |-- statement: string (nullable = true)



#### List of pyspark.sql.Row

In [ ]:

data = [
    Row(a=1, b="string1", c=5.5),
    Row(a=2, b='string2', c=7.0),
    Row(a=3, b='string3', c=0.0),
    Row(a=4, b='string4', c=9.9)
]
df2 = spark.createDataFrame(data)
df2.show(truncate=3) # you can also specify the number of chars when you truncate
df2.printSchema()

+---+---+---+
|  a|  b|  c|
+---+---+---+
|  1|str|5.5|
|  2|str|7.0|
|  3|str|0.0|
|  4|str|9.9|
+---+---+---+

root
 |-- a: long (nullable = true)
 |-- b: string (nullable = true)
 |-- c: double (nullable = true)



## Practice Examples

In [ ]:

Data = [
    ("Simphiwe", 25, "Siteki", 2000),
    ("Bandile", 26, "Manzini", 3000),
    ("Nombuso", 35, "Matsapha", 5000),
    ("Faith", 17, "Kwaluseni", 1000),
    ("Bunmi", 37, "Tubungu", 3500),
    ("Godwin", None, "Mbabane", 5000)
]
df = spark.createDataFrame(Data, ["Firstname", "Age", "City", "Salary"])

print("Content of the original DataFrame")
df.show()

Content of the original DataFrame
+---------+----+---------+------+
|Firstname| Age|     City|Salary|
+---------+----+---------+------+
| Simphiwe|  25|   Siteki|  2000|
|  Bandile|  26|  Manzini|  3000|
|  Nombuso|  35| Matsapha|  5000|
|    Faith|  17|Kwaluseni|  1000|
|    Bunmi|  37|  Tubungu|  3500|
|   Godwin|NULL|  Mbabane|  5000|
+---------+----+---------+------+



In [9]:
print("re-ordering columns alphabetically")
df.select(sorted(df.columns)).show()

[Stage 9:===================>                                       (1 + 2) / 3]

+---+--------+-------+
|Age| Country|   Name|
+---+--------+-------+
| 25|Eswatini|Metfula|
| 28|Zimbabwe| Elliot|
| 30| Namibia|   Aina|
+---+--------+-------+



In [4]:
df.summary().show()

[Stage 4:>                                                          (0 + 1) / 1]

+-------+-------+------------------+--------+
|summary|   Name|               Age| Country|
+-------+-------+------------------+--------+
|  count|      3|                 3|       3|
|   mean|   NULL|27.666666666666668|    NULL|
| stddev|   NULL|2.5166114784235836|    NULL|
|    min|   Aina|                25|Eswatini|
|    25%|   NULL|                25|    NULL|
|    50%|   NULL|                28|    NULL|
|    75%|   NULL|                30|    NULL|
|    max|Metfula|                30|Zimbabwe|
+-------+-------+------------------+--------+



In [11]:
# Select specific columns from the dataframe using select() method

print("Selecting all rows on only these 3 columns")
df.select("Firstname", "Age", "Salary").show()

print("Alternative method of selecting: selecting all rows which meet this condition (all columns)")
df.where(df.Age < 30).show()

selecting evevery row on these 3 columns
+---------+----+------+
|Firstname| Age|Salary|
+---------+----+------+
| Simphiwe|  25|  2000|
|  Bandile|  26|  3000|
|  Nombuso|  35|  5000|
|    Faith|  17|  1000|
|    Bunmi|  37|  3500|
|   Godwin|NULL|  5000|
+---------+----+------+

Alternative method of selecting: selecting all rows which meet this condition (all columns)
+---------+---+---------+------+
|Firstname|Age|     City|Salary|
+---------+---+---------+------+
| Simphiwe| 25|   Siteki|  2000|
|  Bandile| 26|  Manzini|  3000|
|    Faith| 17|Kwaluseni|  1000|
+---------+---+---------+------+



Note:\
In the example above we see two different methods that we can use for selecting rows and columns in a DataFrame.
- First, we can use the `select()` method of a DataFrame to list the columns we want to select. This gives us a ll the rows but only the listed attributes/features are retrieved.
- Second, we can ue the `where()` method to give a condition we want the rows to be retrieved to meet.
Of course you can combine these two to form an `SQL`-like syntax where in you want to retrieve only a certain colums for rows which meet the condition specified on the `where`. You can think on the `select()` method as the `SELECT` clause and the `where()` as the `WHERE` clause.

Let's try it:\
Select the **Age** and **Firstname** for all people whose salary is _greater than_ 3000.

In [16]:
df.select('Age', 'Firstname').where(col('Salary') > 3000).show()

+----+---------+
| Age|Firstname|
+----+---------+
|  35|  Nombuso|
|  37|    Bunmi|
|NULL|   Godwin|
+----+---------+



In [12]:
# filter rows based on a condition
print("Filter age that is greater than 20")
df.filter(col("Age") > 20).show()

Filter age that is greater than 20
+---------+---+--------+------+
|Firstname|Age|    City|Salary|
+---------+---+--------+------+
| Simphiwe| 25|  Siteki|  2000|
|  Bandile| 26| Manzini|  3000|
|  Nombuso| 35|Matsapha|  5000|
|    Bunmi| 37| Tubungu|  3500|
+---------+---+--------+------+



Filtering Rows

An imporatnt data manipulation operation is to be able to filter records according to a certain predicates. For this we can either use the `where()` or `filter()` methods. This is a dupliation provided for the ease of users transitioning from different data processing libraries or engines. When familiar with SQL, you're probably comfortable with `where()`

In [ ]:
df.filter(
    (df.Age > 30) & 
    (df.City=="Tubungu") |
    (df.Salary < 1000)
).show()


+---------+---+-------+------+
|Firstname|Age|   City|Salary|
+---------+---+-------+------+
|    Bunmi| 37|Tubungu|  3500|
+---------+---+-------+------+



With our comparison to **Structured Query Language** forming up, we can now start adding more and more conditions using 
- AND = &
- OR =  |
- NOT = !

Note that these conditions are specified inside the `filter()` method and are connected using the connectors. For readability, each condition is written in a new line in the example above. That is, we have three conditions, one regarding **Age**, the other regarding **City**, and the last one regarding **Salary**.

## Adding, renaming, and removing columns from the DataFrame

You can remove/drop columns from the DataFrame using the `drop()` method. Likewise, you can add new columns to the DF using the `withColumn()` method. Also, you can rename columns using the `withColumnRenamed()` method.

- `drop()` removes a column from the data frame
- `withColumn()` creates a new column. It takes two arguments which are the name of the new column and the expression that creates the column
- `withColumnRenamed()` takes two parameters which are the current name of the column and the new name you want to assign to it
- `alias()` provides a clean and explicit way to name your columns after you've performed work on it

One important thing to note is that these methods are not _"inplace methods"_. By that we mean they do not directly affect the DF so we need to assign the result of the operation into a variable of type `DataFrame`. During this assingment, avoid using the `show()` method because you will then be assigning `None` to the variable since the `show()` method returns `None`.

In [31]:
df = df.withColumn("Bonus", col("Salary") * 0.2)
df.show()

+---------+----+---------+------+------+
|Firstname| Age|     City|Salary| Bonus|
+---------+----+---------+------+------+
| Simphiwe|  25|   Siteki|  2000| 400.0|
|  Bandile|  26|  Manzini|  3000| 600.0|
|  Nombuso|  35| Matsapha|  5000|1000.0|
|    Faith|  17|Kwaluseni|  1000| 200.0|
|    Bunmi|  37|  Tubungu|  3500| 700.0|
|   Godwin|NULL|  Mbabane|  5000|1000.0|
+---------+----+---------+------+------+



Here, we add a new column to the data frame which is called Bonus and it is deviced from Salary. The value is 20 percent of the Salary.

In [32]:
df = df.withColumn("Region", lit("Manzini"))
df.show()

+---------+----+---------+------+------+-------+
|Firstname| Age|     City|Salary| Bonus| Region|
+---------+----+---------+------+------+-------+
| Simphiwe|  25|   Siteki|  2000| 400.0|Manzini|
|  Bandile|  26|  Manzini|  3000| 600.0|Manzini|
|  Nombuso|  35| Matsapha|  5000|1000.0|Manzini|
|    Faith|  17|Kwaluseni|  1000| 200.0|Manzini|
|    Bunmi|  37|  Tubungu|  3500| 700.0|Manzini|
|   Godwin|NULL|  Mbabane|  5000|1000.0|Manzini|
+---------+----+---------+------+------+-------+



Here, we create a new column called Region and set the value fore everyone to be Manzini using the `lit()` function

In [33]:
df = df.withColumnRenamed("Region", "Sigodzi")
df.show()
df.printSchema()

+---------+----+---------+------+------+-------+
|Firstname| Age|     City|Salary| Bonus|Sigodzi|
+---------+----+---------+------+------+-------+
| Simphiwe|  25|   Siteki|  2000| 400.0|Manzini|
|  Bandile|  26|  Manzini|  3000| 600.0|Manzini|
|  Nombuso|  35| Matsapha|  5000|1000.0|Manzini|
|    Faith|  17|Kwaluseni|  1000| 200.0|Manzini|
|    Bunmi|  37|  Tubungu|  3500| 700.0|Manzini|
|   Godwin|NULL|  Mbabane|  5000|1000.0|Manzini|
+---------+----+---------+------+------+-------+

root
 |-- Firstname: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- City: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Bonus: double (nullable = true)
 |-- Sigodzi: string (nullable = false)



In [34]:
df=df.drop("Sigodzi")
df.show()

+---------+----+---------+------+------+
|Firstname| Age|     City|Salary| Bonus|
+---------+----+---------+------+------+
| Simphiwe|  25|   Siteki|  2000| 400.0|
|  Bandile|  26|  Manzini|  3000| 600.0|
|  Nombuso|  35| Matsapha|  5000|1000.0|
|    Faith|  17|Kwaluseni|  1000| 200.0|
|    Bunmi|  37|  Tubungu|  3500| 700.0|
|   Godwin|NULL|  Mbabane|  5000|1000.0|
+---------+----+---------+------+------+



## Exercise 1
Create a new column called **New Salary** based on the addition of `Salary` and `Bonus`
Hint: 
 - New Salary = Salary + Bonus
 - `col()`

In [36]:
## Your solution here

## Exercise 2

Rename the `Firstname` column to `Name`

In [ ]:
## Your solution here

## Update Columns based on Conditions

- Change the city name for **Nombuso** to Ludzeludze

In [41]:
df = df\
    .withColumn("City", when(col("Firstname") == "Nombuso", lit("Ludzeludze"))\
    .otherwise(col("City")))

df.show()

+---------+----+----------+------+------+
|Firstname| Age|      City|Salary| Bonus|
+---------+----+----------+------+------+
| Simphiwe|  25|    Siteki|  2000| 400.0|
|  Bandile|  26|   Manzini|  3000| 600.0|
|  Nombuso|  35|Ludzeludze|  5000|1000.0|
|    Faith|  17| Kwaluseni|  1000| 200.0|
|    Bunmi|  37|   Tubungu|  3500| 700.0|
|   Godwin|NULL|   Mbabane|  5000|1000.0|
+---------+----+----------+------+------+



## Exercise 3

Change the Manzini City to Mbabane for all people who reside in Manzini.\
Hint
- First get/select the people
- Change only their City

In [ ]:
## Your solution here

## Groupings and Aggregations

The easiest way to count record occurence is to use the `groupBy()` method, passing the columns we wish to group on, as a parameter. A grouped by data frame is not a dataframe anymore, instead, it becomes a `GroupedData` object. You can't inspect it using `show()` but it's waitng for further instruction to become show-able again.

The `agg()` method also accepts a dictionary in the form `{col_name: agg_function}` where both are strings. This approach does not allow you to `alias()` the columns, instead, we could use aggreagate functions from `pyspark.sql.functions`. `agg()` takes an arbitrary number of aggreagate functions.

After the application of the aggregate function on the `GroupedData` object, we're back with a data frame and now we can use the `orderBy()` method to order the data by decreasing or increasing order of a particular columns.

In [43]:
print("average salary by city")

df.groupBy("City").agg(
    avg("Salary").alias("Avg_Salary"), 
    count('*').alias("Num_employees")
).show()

[Stage 48:===========================================>              (3 + 1) / 4]

+----------+----------+-------------+
|      City|Avg_Salary|Num_employees|
+----------+----------+-------------+
|    Siteki|    2000.0|            1|
|Ludzeludze|    5000.0|            1|
|   Manzini|    3000.0|            1|
| Kwaluseni|    1000.0|            1|
|   Mbabane|    5000.0|            1|
|   Tubungu|    3500.0|            1|
+----------+----------+-------------+



In [45]:
print("Sort age in descending order")

df.orderBy(col("Age").desc()).show()

Sort age in descending order
+---------+----+----------+------+------+
|Firstname| Age|      City|Salary| Bonus|
+---------+----+----------+------+------+
|    Bunmi|  37|   Tubungu|  3500| 700.0|
|  Nombuso|  35|Ludzeludze|  5000|1000.0|
|  Bandile|  26|   Manzini|  3000| 600.0|
| Simphiwe|  25|    Siteki|  2000| 400.0|
|    Faith|  17| Kwaluseni|  1000| 200.0|
|   Godwin|NULL|   Mbabane|  5000|1000.0|
+---------+----+----------+------+------+



Note:\
This same operation can be done by using the `desc()` function instead of `col().desc()` but the results are pretty much the same thing.

In [ ]:
df.orderBy(desc("Age")).show()

+---------+----+----------+------+------+
|Firstname| Age|      City|Salary| Bonus|
+---------+----+----------+------+------+
|    Bunmi|  37|   Tubungu|  3500| 700.0|
|  Nombuso|  35|Ludzeludze|  5000|1000.0|
|  Bandile|  26|   Manzini|  3000| 600.0|
| Simphiwe|  25|    Siteki|  2000| 400.0|
|    Faith|  17| Kwaluseni|  1000| 200.0|
|   Godwin|NULL|   Mbabane|  5000|1000.0|
+---------+----+----------+------+------+



## Handling Missing Values

- Give a specific value for a column
- Use the same value across the dataset
- Dropping missing values
- Use `Imputer` and measures of central tendency

In [48]:
df.fillna({"Age":0}).show() # for age, fill with 0

+---------+---+----------+------+------+
|Firstname|Age|      City|Salary| Bonus|
+---------+---+----------+------+------+
| Simphiwe| 25|    Siteki|  2000| 400.0|
|  Bandile| 26|   Manzini|  3000| 600.0|
|  Nombuso| 35|Ludzeludze|  5000|1000.0|
|    Faith| 17| Kwaluseni|  1000| 200.0|
|    Bunmi| 37|   Tubungu|  3500| 700.0|
|   Godwin|  0|   Mbabane|  5000|1000.0|
+---------+---+----------+------+------+



In [49]:
df.fillna(0).show() # fill using 0 across the dataset

+---------+---+----------+------+------+
|Firstname|Age|      City|Salary| Bonus|
+---------+---+----------+------+------+
| Simphiwe| 25|    Siteki|  2000| 400.0|
|  Bandile| 26|   Manzini|  3000| 600.0|
|  Nombuso| 35|Ludzeludze|  5000|1000.0|
|    Faith| 17| Kwaluseni|  1000| 200.0|
|    Bunmi| 37|   Tubungu|  3500| 700.0|
|   Godwin|  0|   Mbabane|  5000|1000.0|
+---------+---+----------+------+------+



In [50]:
df.na.drop().show() # drop missing values

+---------+---+----------+------+------+
|Firstname|Age|      City|Salary| Bonus|
+---------+---+----------+------+------+
| Simphiwe| 25|    Siteki|  2000| 400.0|
|  Bandile| 26|   Manzini|  3000| 600.0|
|  Nombuso| 35|Ludzeludze|  5000|1000.0|
|    Faith| 17| Kwaluseni|  1000| 200.0|
|    Bunmi| 37|   Tubungu|  3500| 700.0|
+---------+---+----------+------+------+



In [ ]:
imputer = Imputer(
    inputCols=['Age'],
    outputCols=['Age1'],
    strategy='mean'
)
model = imputer.fit(df)
df1 = model.transform(df)
df1.show()

+---------+----+----------+------+------+----+
|Firstname| Age|      City|Salary| Bonus|Age1|
+---------+----+----------+------+------+----+
| Simphiwe|  25|    Siteki|  2000| 400.0|  25|
|  Bandile|  26|   Manzini|  3000| 600.0|  26|
|  Nombuso|  35|Ludzeludze|  5000|1000.0|  35|
|    Faith|  17| Kwaluseni|  1000| 200.0|  17|
|    Bunmi|  37|   Tubungu|  3500| 700.0|  37|
|   Godwin|NULL|   Mbabane|  5000|1000.0|  28|
+---------+----+----------+------+------+----+



## Handle Duplicates

In [53]:
df.select("Firstname").distinct().show()

[Stage 66:===========================================>              (3 + 1) / 4]

+---------+
|Firstname|
+---------+
| Simphiwe|
|  Bandile|
|  Nombuso|
|    Faith|
|   Godwin|
|    Bunmi|
+---------+



In [54]:
df.dropDuplicates(['Salary']).show()

+---------+---+----------+------+------+
|Firstname|Age|      City|Salary| Bonus|
+---------+---+----------+------+------+
|    Faith| 17| Kwaluseni|  1000| 200.0|
| Simphiwe| 25|    Siteki|  2000| 400.0|
|  Bandile| 26|   Manzini|  3000| 600.0|
|    Bunmi| 37|   Tubungu|  3500| 700.0|
|  Nombuso| 35|Ludzeludze|  5000|1000.0|
+---------+---+----------+------+------+



## Joining Data

At its most basic, a join operation is a way to take the data from a data frame and add it to another one according to a set of rules, more like SQL joins. The join operation has three major ingredients

- Two tables, called left and right tables
- one or more predicates which are the series of conditions that deterin if records between the two tables are joined
- A method to indicate how to perform the join when the predicate succeeds and when it fails.

Syntax

```python
[LEFT].join(
    [RIGHT],
    on=[PREDICATES],
    how=[METHOD]
)
```

A join is performed on two tables, called left and right, at a time. The predicates of a PySpark join are rules between columns of the left and right data frames. A join is performed record-wise, where each record on the left data frame is compared (via the predicates) to each record on the right data frame. If the predicates return `True`, the join is a match and is a failure if `False`. Usually, in SQL the predicate is formed by keys on the two tables; primarily the primary and secondary keys.

You are not limited to a single test in your predicates. You can use multiple conditions by separating them with Boolean operator such as `|`, or `&`. You can also use different comparison operators, operator is an element of `{<, <=, >, >=< !=/<>, ==}`.

If you have multiple predicates such as `(left['col1'] == right['col2']) & (left['col3'] > right['col4'])`, you can put them into a list such as `[left['col1'] == right['col2'], left['col3'] > right['col4']]`. This make your intent more explicit and avoid the parenthesis headache. Also, if you're performing an "equi-join", where you have testing for equality between identiical column names, you can just have the name of the columns as a string or list of strings as the predicate.

```python
left.join(
    right,
    on='column' # ['col1', 'col1'],
    method=[METHOD]
)
```

A join method boils down to two questions:
1. What happens when the return value of the predicate is True?
2. What happens when the return value of the predicate is False?

### Cross Join

A cross join is the nuclear option. It returns a record for every record pair, regardless of the value that the predicate returns. `how="cross"`. It combines every row on the left with every row on the right more like a Cartesian Product. The number of rows returned, n, is `p*q` where p is the number of records on the left table, and q is the number of records on the right table.

### Inner Join

PySpark will default to an inner join if you don't pass the method explicitly. It returns a record if the predicates is True and drops it if it is False. In set theory, it returns the intersection of the left and right tables; only the data that is common between the two tables, as per the prediates. `how=inner`.


### Left and Right Outer Joins

Left, `how=left` or `how=left_outer`, and right `how=right` or `how=right_outer`, are like inner join, in the sense that they generate a record for a successful predicates. The difference is what happens when the predicate is false.

- A left join will add the unmatched records from the left table in the joined table, filling the columns coming from the right table with NULL
- A right will add the unmatched columns coming from the right in the joined table, filling the columns comig from the left table with NULL

In practice, it means that your joined table is guaranteed to contain all records of the table which feed the join, main table.

### Full Outer Join

A full outer, `how=outer` or `how=full` or `how=full_outer`, join is simply the fusion of the left and right join. It will add unmatched records from the left and right table, padding with NULL.

### Left Semi-join and Left Anti-join

A left semi-join, `how="left_semi"` is the same as an inner join but only keeps the column in the left table. It also won't duplicate the records in the left tablw if they fulfil the predicate with more than one record in the right table. A right anti-join is the opposite of an inner join. It will keep only the records that do not match the predicate with any record in the right table, dropping any successful match.

